In [1]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error
from datetime import datetime, timedelta
import pytz

try:
    os.getcwd()
except FileNotFoundError:
    os.chdir(os.path.expanduser("~"))

BASE_DIR = '/Users/abdrhmn/projects/weather-data-pipeline'
ASSETS_DIR = os.path.join(BASE_DIR, 'assets')
WEATHER_CSV_PATH = os.path.join(ASSETS_DIR, 'weather.csv')

print(f"Project path : {BASE_DIR}")
print(f"CSV path     : {WEATHER_CSV_PATH}")
print(f"File exists? : {os.path.exists(WEATHER_CSV_PATH)}")

env_path = os.path.join(os.path.expanduser("~"), '.env')
load_dotenv(env_path)
API_KEY = os.getenv("WEATHER_API_KEY")
BASE_URL = "https://api.openweathermap.org/data/2.5/"
print(f"API Key ready: {API_KEY[:10]}...")


def get_current_weather(city):
    try:
        url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return {
            'city': data['name'],
            'current_temp': round(data['main']['temp']),
            'feels_like': round(data['main']['feels_like']),
            'temp_min': round(data['main']['temp_min']),
            'temp_max': round(data['main']['temp_max']),
            'humidity': data['main']['humidity'],
            'description': data['weather'][0]['description'],
            'country': data['sys']['country'],
            'WindGustDir': data['wind'].get('deg', 0),
            'Pressure': data['main'].get('pressure', 0),
            'Wind_Gust_Speed': data['wind'].get('speed', 0),
            'WindGust': data['wind'].get('gust', None)
        }
    except Exception as e:
        print(f"Error fetching weather for {city}: {e}")
        return None


def read_historical_data(filename):
    try:
        df = pd.read_csv(filename)
        print(f"Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")
        df = df.dropna()
        df = df.drop_duplicates()  # <--- THIS WAS THE BUG (was "df.s()")
        df = df.reset_index(drop=True)
        print(f"After cleaning: {df.shape[0]} rows")
        return df
    except FileNotFoundError:
        print(f"File not found: {filename}")
        raise


def prepare_data(data):
    le = LabelEncoder()
    data['WindGustDir'] = le.fit_transform(data['WindGustDir'].astype(str))
    data['RainTomorrow'] = le.fit_transform(data['RainTomorrow'].astype(str))
    X = data.drop('RainTomorrow', axis=1)
    y = data['RainTomorrow']
    return X, y, le


def train_rain_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("\nRain model report:")
    print(classification_report(y_test, y_pred))
    return model


def prepare_regression_data(data, feature):
    X, y = [], []
    values = data[feature].values
    for i in range(len(values) - 1):
        X.append(values[i])
        y.append(values[i + 1])
    return np.array(X).reshape(-1, 1), np.array(y)


def train_regression_model(X, y):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)
    return model


def predict_future(model, current_value, steps=5):
    predictions = [current_value]
    for _ in range(steps):
        next_val = model.predict(np.array([[predictions[-1]]]))[0]
        predictions.append(round(float(next_val), 2))
    return predictions[1:]


def weather_view():
    arabic_cities = [
        "Cairo", "Riyadh", "Dubai", "Baghdad", "Beirut",
        "Amman", "Kuwait", "Doha", "Casablanca", "Tunis"
    ]

    try:
        historical_data = read_historical_data(WEATHER_CSV_PATH)
    except FileNotFoundError:
        print(f"Could not find file at: {WEATHER_CSV_PATH}")
        return None

    print("\nTraining models...")
    X, y, le = prepare_data(historical_data)
    rain_model = train_rain_model(X, y)

    X_temp, y_temp = prepare_regression_data(historical_data, 'Temp')
    X_hum, y_hum = prepare_regression_data(historical_data, 'Humidity')
    temp_model = train_regression_model(X_temp, y_temp)
    hum_model = train_regression_model(X_hum, y_hum)
    print("Models trained successfully!\n")

    all_results = []

    for city in arabic_cities:
        print(f"\nFetching data for {city}...")
        current_weather = get_current_weather(city)
        if not current_weather:
            print(f"Failed to get data for {city}")
            continue

        wind_deg = current_weather.get('WindGustDir', 0) % 360
        compass_points = [
            ("N", 348.75, 360), ("N", 0, 11.25),
            ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25),
            ("ENE", 56.25, 78.75), ("E", 78.75, 101.25),
            ("ESE", 101.25, 123.75), ("SE", 123.75, 146.25),
            ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
            ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25),
            ("WSW", 236.25, 258.75), ("W", 258.75, 281.25),
            ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
            ("NNW", 326.25, 348.75),
        ]

        compass_direction = "N"
        for direction, start, end in compass_points:
            if start <= wind_deg < end:
                compass_direction = direction
                break

        if compass_direction in le.classes_:
            compass_direction_encoded = le.transform([compass_direction])[0]
        else:
            compass_direction_encoded = 0

        current_data = {
            'MinTemp': current_weather.get('temp_min', 0),
            'MaxTemp': current_weather.get('temp_max', 0),
            'Humidity': current_weather.get('humidity', 0),
            'WindGustDir': compass_direction_encoded,
            'WindGustSpeed': current_weather.get('Wind_Gust_Speed', 0),
            'Pressure': current_weather.get('Pressure', 0),
            'Temp': current_weather.get('current_temp', 0)
        }
        current_df = pd.DataFrame([current_data])

        try:
            rain_prediction = rain_model.predict(current_df)[0]
            if hasattr(rain_model, 'predict_proba'):
                rain_probability = rain_model.predict_proba(current_df)[0][1] * 100
            else:
                rain_probability = float(rain_prediction) * 100
        except Exception as e:
            print(f"Rain prediction error for {city}: {e}")
            rain_probability = 0

        future_temp = predict_future(
            temp_model, current_weather.get('current_temp', 0)
        )
        future_hum = predict_future(
            hum_model, current_weather.get('humidity', 0)
        )

        result = {
            'city': city,
            'country': current_weather.get('country'),
            'current_temp': current_weather.get('current_temp'),
            'temp_max': current_weather.get('temp_max'),
            'temp_min': current_weather.get('temp_min'),
            'humidity': current_weather.get('humidity'),
            'description': current_weather.get('description'),
            'wind_direction': compass_direction,
            'rain_probability': rain_probability,
            'future_temp': future_temp,
            'future_humidity': future_hum
        }
        all_results.append(result)

        print("\n" + "=" * 55)
        print(f" {city} | {current_weather.get('country')}")
        print("=" * 55)
        print(f"  Temp        : {current_weather.get('current_temp')}C")
        print(f"  Max Temp    : {current_weather.get('temp_max')}C")
        print(f"  Min Temp    : {current_weather.get('temp_min')}C")
        print(f"  Humidity    : {current_weather.get('humidity')}%")
        print(f"  Description : {current_weather.get('description')}")
        print(f"  Wind Dir    : {compass_direction}")
        print(f"  Rain Prob   : {rain_probability:.2f}%")
        print(f"  Future Temp : {future_temp}")
        print(f"  Future Hum  : {future_hum}")
        print("=" * 55)

    return all_results


results = weather_view()

if results:
    timezone = pytz.timezone('Africa/Cairo')
    now = datetime.now(timezone)
    next_hour = (now + timedelta(hours=1)).replace(
        minute=0, second=0, microsecond=0
    )
    future_times = [next_hour + timedelta(hours=i) for i in range(5)]

    print("\n\n" + "=" * 75)
    print("Weather Comparison - Arabic Cities")
    print(f"Time: {now.strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 75)
    print(
        f"{'City':<15}{'Temp':^10}{'Humidity':^10}"
        f"{'Rain Prob':^18}{'Wind Dir':^12}"
    )
    print("-" * 75)

    for r in results:
        print(
            f"  {r['city']:<13}"
            f"{str(r['current_temp'])+'C':^10}"
            f"{str(r['humidity'])+'%':^10}"
            f"{str(round(r['rain_probability'], 1))+'%':^18}"
            f"{r['wind_direction']:^12}"
        )

    print("=" * 75)

    print("\n\nFuture Forecasts:")
    print("=" * 75)

    for r in results:
        print(f"\n {r['city']}:")
        print(f"{'Hour':<10}{'Time':<22}{'Temp':^10}{'Humidity':^10}")
        print("-" * 55)
        for i, (time, temp, hum) in enumerate(
            zip(future_times, r['future_temp'], r['future_humidity']), 1
        ):
            print(
                f"  {i:<8}"
                f"{time.strftime('%Y-%m-%d %H:%M'):<22}"
                f"{str(temp)+'C':^10}"
                f"{str(hum)+'%':^10}"
            )
else:
    print("Weather analysis failed.")

Project path : /Users/abdrhmn/projects/weather-data-pipeline
CSV path     : /Users/abdrhmn/projects/weather-data-pipeline/assets/weather.csv
File exists? : True
API Key ready: c7f610680d...
Loaded data: 366 rows, 8 columns
After cleaning: 363 rows

Training models...

Rain model report:
              precision    recall  f1-score   support

           0       0.85      0.98      0.91        57
           1       0.86      0.38      0.52        16

    accuracy                           0.85        73
   macro avg       0.85      0.68      0.72        73
weighted avg       0.85      0.85      0.83        73

Models trained successfully!


Fetching data for Cairo...
Rain prediction error for Cairo: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.


 Cairo | EG
  Temp        : 24C
  Max Temp    : 24C
  Min Temp    : 24C
  Humidity    : 29%
  Description : few clouds
  Wind Dir    : WNW
  Rain Prob   : 0.00%
  Fu